# Correlation and linear regression in clinical and public health research with computational implementation in R and Python

**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa

## 1) Purpose

This notebook provides a clear, step-by-step analytical guide for evaluating the relationship between two continuous clinical variables using **correlation** and **linear regression**.

The applied objective is to use the attached clinical dataset to answer the following questions:

1. Is patient age linearly associated with serum creatinine at ICU admission?
2. How much does mean serum creatinine change, on average, for each additional year of age?
3. How should the statistical estimates be interpreted in clinical and epidemiological terms?

## 2) Required Libraries

This notebook uses standard scientific Python libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

# Display tables with a reasonable number of decimals.
pd.set_option("display.precision", 4)

## 3) Data Import (Excel)

This guide uses the Excel file **`data.xlsx`** with a sheet named **`Data`**.

- If `data.xlsx` is in the same folder as this notebook, you can keep the path as `"data.xlsx"`.
- Otherwise, replace with the full path to your file.

In [ ]:
# Update the path if needed
excel_path = "data.xlsx"
sheet_name = "Data"

data = pd.read_excel(excel_path, sheet_name=sheet_name)

# Quick inspection
data.head(), data.shape

## 4) Variable definitions and data inspection

Before conducting statistical analysis, the investigator should inspect the structure of the dataset, confirm the variable names, and review the first rows.

The main variables used in this report are:

- `age`: patient age at ICU admission, measured in years.
- `creatinine`: serum creatinine level, measured in mg/dL.
- `saps_3`: Simplified Acute Physiology Score 3 at ICU admission.
- `chronic_kidney_disease`: indicator of chronic kidney dysfunction, coded as 0 = no and 1 = yes.
- `vasopressor`: use of any vasopressor at ICU admission, coded as 0 = no and 1 = yes.
- `outcome`: vital status at hospital discharge, coded as 0 = alive and 1 = dead.

In [ ]:
# Number of rows and columns.
print("Dataset dimensions:", data.shape)

# Variable names.
print("\nVariable names:")
print(list(data.columns))

# First six records.
data.head(6)

## 5) Basic data preparation

The primary analysis requires complete observations for `age` and `creatinine`. The adjusted analysis also requires complete observations for `saps_3`, `chronic_kidney_disease`, and `vasopressor`.

The variables are converted to appropriate types before complete-case selection. This prevents retaining values that may become missing during numeric conversion.

\[
D = \{(X_i, Y_i): X_i \text{ and } Y_i \text{ are both observed}\}
\]

In [ ]:
# Start from the imported dataset.
analysis_data = data.copy()

# Convert continuous variables to numeric values.
for column in ["age", "creatinine", "saps_3"]:
    analysis_data[column] = pd.to_numeric(analysis_data[column], errors="coerce")

# Convert binary clinical indicators to numeric values first.
# This step is useful if the variables were imported as text.
for column in ["chronic_kidney_disease", "vasopressor"]:
    analysis_data[column] = pd.to_numeric(analysis_data[column], errors="coerce")

# Remove records with missing values in any variable used in the analyses.
analysis_variables = [
    "age",
    "creatinine",
    "saps_3",
    "chronic_kidney_disease",
    "vasopressor",
]
analysis_data = analysis_data.dropna(subset=analysis_variables).copy()

# Convert binary indicators to categorical labels for clearer regression output.
# The reference category will be "No" in the adjusted model.
analysis_data["chronic_kidney_disease"] = (
    analysis_data["chronic_kidney_disease"].map({0: "No", 1: "Yes"})
)
analysis_data["vasopressor"] = (
    analysis_data["vasopressor"].map({0: "No", 1: "Yes"})
)

# Confirm the analytic sample size and observed age range.
n = len(analysis_data)
print("Analytic sample size:", n)
print("Observed age range:", analysis_data["age"].min(), "to", analysis_data["age"].max())

# Summarize the two primary continuous variables.
analysis_data[["age", "creatinine"]].describe()

## 6) Graphical inspection before modeling

A scatterplot should precede correlation and regression. The purpose is to evaluate whether the association appears approximately linear and whether there are influential observations, restricted ranges, subgroups, or unusual values.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(analysis_data["age"], analysis_data["creatinine"])
plt.xlabel("Age at ICU admission (years)")
plt.ylabel("Serum creatinine (mg/dL)")
plt.title("Scatterplot of Age and Serum Creatinine")
plt.show()

## 7) Pearson correlation

The value of Pearson's correlation coefficient ranges from -1 to +1.

- Values near +1 indicate a strong positive linear association.
- Values near -1 indicate a strong negative linear association.
- Values near 0 indicate little or no linear association.

Correlation does not establish causality and does not quantify agreement between two measurement methods.

Clinical interpretation should focus on direction, magnitude, uncertainty, and plausibility.

A positive Pearson correlation suggests that older patients tend to have higher creatinine values. A negative value would suggest that older patients tend to have lower creatinine values. A correlation close to zero does not exclude nonlinear relationships or subgroup-specific patterns.

In this analysis, the correlation coefficient should be interpreted as a measure of **linear association**, not as a causal effect and not as a regression slope.

In [ ]:
# Pearson correlation coefficient and two-sided P value.
r_value, r_p_value = stats.pearsonr(analysis_data["age"], analysis_data["creatinine"])

# 95% confidence interval for Pearson correlation using Fisher's z transformation.
# z = arctanh(r), SE(z) = 1 / sqrt(n - 3)
z_value = np.arctanh(r_value)
z_se = 1 / np.sqrt(n - 3)
z_critical = stats.norm.ppf(0.975)

r_ci_low = np.tanh(z_value - z_critical * z_se)
r_ci_high = np.tanh(z_value + z_critical * z_se)

correlation_summary = pd.DataFrame({
    "measure": [
        "Pearson correlation coefficient",
        "95% CI lower bound",
        "95% CI upper bound",
        "P value",
    ],
    "value": [r_value, r_ci_low, r_ci_high, r_p_value],
})

correlation_summary

## 9) Fitting the linear regression model


In [ ]:
simple_model = smf.ols("creatinine ~ age", data=analysis_data).fit()

simple_model.summary()

## 10) Regression Coefficients and 95% Confidence Intervals

The regression coefficient for `age` estimates the expected mean change in serum creatinine, in mg/dL, for each additional year of age.

The intercept estimates the expected mean creatinine when age equals zero. In this clinical context, age zero is outside the relevant adult ICU population and should not receive a substantive clinical interpretation.

In [ ]:
# Coefficient estimates, standard errors, P values, and confidence intervals.
coef_table = simple_model.summary2().tables[1]
coef_table = coef_table.rename(columns={
    "Coef.": "estimate",
    "Std.Err.": "standard_error",
    "P>|t|": "p_value",
    "[0.025": "ci_low",
    "0.975]": "ci_high",
})

simple_regression_results = (
    coef_table[["estimate", "standard_error", "p_value", "ci_low", "ci_high"]]
    .reset_index()
    .rename(columns={"index": "term"})
)

simple_regression_results

## 11) Coefficient of determination

In [ ]:
r_squared_model = simple_model.rsquared
r_squared_from_correlation = r_value ** 2

r_squared_summary = pd.DataFrame({
    "measure": ["R-squared from regression", "Squared Pearson correlation"],
    "value": [r_squared_model, r_squared_from_correlation],
})

r_squared_summary

## 12) Fitted values and residuals
Residuals are the observed deviations from the fitted regression line.

In [ ]:
analysis_data["fitted_creatinine"] = simple_model.fittedvalues
analysis_data["residual"] = simple_model.resid

columns_to_display = ["age", "creatinine", "fitted_creatinine", "residual"]
if "patient" in analysis_data.columns:
    columns_to_display = ["patient"] + columns_to_display

analysis_data[columns_to_display].head()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(analysis_data["age"], analysis_data["creatinine"])

# Create a smooth sequence of ages to draw the fitted regression line.
age_grid = np.linspace(analysis_data["age"].min(), analysis_data["age"].max(), 100)
predicted_grid = simple_model.predict(pd.DataFrame({"age": age_grid}))

plt.plot(age_grid, predicted_grid, linewidth=2)
plt.xlabel("Age at ICU admission (years)")
plt.ylabel("Serum creatinine (mg/dL)")
plt.title("Simple Linear Regression of Creatinine on Age")
plt.show()

## 13) Model diagnostics

Linear regression should not be interpreted only from the coefficient table. Diagnostic plots help evaluate model assumptions and influential observations.

Interpretation of diagnostic plots:

- The residuals-versus-fitted plot helps assess nonlinearity and unequal variance.
- The normal Q-Q plot helps evaluate whether residuals are approximately normally distributed.
- The scale-location plot helps evaluate heteroscedasticity.
- The residuals-versus-leverage plot helps identify influential observations.

In [ ]:
# Residuals versus fitted values.
plt.figure(figsize=(7, 5))
plt.scatter(simple_model.fittedvalues, simple_model.resid)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals Versus Fitted Values")
plt.show()

In [ ]:
# Normal Q-Q plot of residuals.
sm.qqplot(simple_model.resid, line="45", fit=True)
plt.title("Normal Q-Q Plot of Residuals")
plt.show()

In [ ]:
# Scale-location plot.
standardized_residuals = simple_model.get_influence().resid_studentized_internal
sqrt_abs_standardized_residuals = np.sqrt(np.abs(standardized_residuals))

plt.figure(figsize=(7, 5))
plt.scatter(simple_model.fittedvalues, sqrt_abs_standardized_residuals)
plt.xlabel("Fitted values")
plt.ylabel("Square root of absolute standardized residuals")
plt.title("Scale-Location Plot")
plt.show()

In [ ]:
# Residuals versus leverage.
influence = simple_model.get_influence()
leverage = influence.hat_matrix_diag

plt.figure(figsize=(7, 5))
plt.scatter(leverage, standardized_residuals)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Leverage")
plt.ylabel("Standardized residuals")
plt.title("Residuals Versus Leverage")
plt.show()

## 14) Adjusted linear regression

Clinical and public health analyses often require adjustment for other measured factors. In this example, creatinine is modeled as a function of age, severity of illness, chronic kidney disease, and vasopressor use.

The binary variables `chronic_kidney_disease` and `vasopressor` are represented as categorical variables with `No` as the reference category.

In [ ]:
adjusted_model = smf.ols(
    "creatinine ~ age + saps_3 + "
    "C(chronic_kidney_disease, Treatment(reference='No')) + "
    "C(vasopressor, Treatment(reference='No'))",
    data=analysis_data,
).fit()

adjusted_model.summary()

In [ ]:
adj_coef_table = adjusted_model.summary2().tables[1]
adj_coef_table = adj_coef_table.rename(columns={
    "Coef.": "estimate",
    "Std.Err.": "standard_error",
    "P>|t|": "p_value",
    "[0.025": "ci_low",
    "0.975]": "ci_high",
})

adjusted_regression_results = (
    adj_coef_table[["estimate", "standard_error", "p_value", "ci_low", "ci_high"]]
    .reset_index()
    .rename(columns={"index": "term"})
)

adjusted_regression_results

### Clinical Interpretation of the Adjusted Model

The adjusted model estimates the association between age and creatinine conditional on SAPS 3, chronic kidney disease, and vasopressor use.

Because this is an illustrative observational-style analysis of a synthetic dataset, the adjusted coefficient should be interpreted as a conditional association rather than a causal effect. Adjustment may change the magnitude of the age coefficient because the model compares patients of the same severity score and the same categories of chronic kidney disease and vasopressor use.

## 15) Predicted Mean Creatinine for Selected Ages

These predictions should be interpreted only within the observed or clinically plausible range of age in the dataset.

The code below uses confidence intervals for the expected mean creatinine, not prediction intervals for individual patients.

These intervals estimate uncertainty around the expected mean creatinine at each selected age. They should not be interpreted as the expected range for an individual patient's creatinine value.

In [ ]:
new_patients = pd.DataFrame({"age": [40, 60, 80]})

prediction_summary = simple_model.get_prediction(new_patients).summary_frame(alpha=0.05)

prediction_table = pd.concat(
    [new_patients, prediction_summary[["mean", "mean_ci_lower", "mean_ci_upper"]]],
    axis=1,
)

prediction_table = prediction_table.rename(columns={
    "mean": "predicted_mean_creatinine",
    "mean_ci_lower": "ci_low",
    "mean_ci_upper": "ci_high",
})

prediction_table

In [ ]:
# Example only: prediction intervals for individual patients can be requested as follows.
individual_prediction_table = pd.concat(
    [new_patients, prediction_summary[["obs_ci_lower", "obs_ci_upper"]]],
    axis=1,
)

individual_prediction_table = individual_prediction_table.rename(columns={
    "obs_ci_lower": "prediction_interval_low",
    "obs_ci_upper": "prediction_interval_high",
})

individual_prediction_table

## 16) Final Summary Table

The final table summarizes the main findings from the correlation, simple regression, and adjusted regression analyses.

In [ ]:
def format_p(p):
    """Format P values for a clinical research table."""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

slope_age = simple_regression_results.loc[
    simple_regression_results["term"] == "age"
].iloc[0]

# In statsmodels, the age term remains named "age" in the adjusted model.
adj_slope_age = adjusted_regression_results.loc[
    adjusted_regression_results["term"] == "age"
].iloc[0]

final_summary_table = pd.DataFrame({
    "analysis": [
        "Pearson correlation: age and creatinine",
        "Simple regression slope: creatinine on age",
        "Simple regression R-squared",
        "Adjusted regression slope for age",
    ],
    "estimate": [
        f"{r_value:.3f}",
        f"{slope_age['estimate']:.4f}",
        f"{r_squared_model:.3f}",
        f"{adj_slope_age['estimate']:.4f}",
    ],
    "confidence_interval": [
        f"95% CI {r_ci_low:.3f} to {r_ci_high:.3f}",
        f"95% CI {slope_age['ci_low']:.4f} to {slope_age['ci_high']:.4f}",
        "Not applicable",
        f"95% CI {adj_slope_age['ci_low']:.4f} to {adj_slope_age['ci_high']:.4f}",
    ],
    "p_value": [
        format_p(r_p_value),
        format_p(slope_age["p_value"]),
        "Not applicable",
        format_p(adj_slope_age["p_value"]),
    ],
})

final_summary_table